 Google_Play_Store_Analytics

Import Libraries

In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

In [ ]:
nltk.download('vader_lexicon')

Load Dataset

In [ ]:
apps_df=pd.read_csv("/content/Play Store Data.csv")
reviews_df=pd.read_csv("/content/User Reviews.csv")

In [ ]:
apps_df

In [ ]:
reviews_df

Data Cleaning

In [ ]:
apps_df.isna().sum()

In [ ]:
apps_df=apps_df.dropna(subset=['Rating'])


In [ ]:
apps_df.isna().sum()

In [ ]:
apps_df.columns

In [ ]:
for column in apps_df.columns:
  apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)


In [ ]:
apps_df.isna().sum()

In [ ]:
apps_df.duplicated().sum()

In [ ]:
apps_df.drop_duplicates(inplace=True)

In [ ]:
apps_df.duplicated().sum()

In [ ]:
apps_df=apps_df[apps_df["Rating"]<=5]

In [ ]:
reviews_df.dropna(subset=["Translated_Review"],inplace=True)

In [ ]:
merged_df=pd.merge(apps_df,reviews_df,on ='App',how='inner')

In [ ]:
merged_df

In [ ]:
apps_df['Reviews']= apps_df["Reviews"].astype(int)


In [ ]:
apps_df["Price"]=apps_df["Price"].str.replace("$","").astype(float)

In [ ]:
apps_df["Installs"] = apps_df["Installs"].str.replace(",", "").str.replace("+", "").astype(int)

In [ ]:
def convert_size(size):
  if "M" in size:
    return float(size.replace("M",""))
  elif "K" in size:
    return float(size.replace("K","")/1024)
  else:
    return np.nan



In [ ]:
apps_df["Size"]=apps_df["Size"].apply(convert_size)

In [ ]:
apps_df["Size"]

In [ ]:
apps_df["Log_Installs"]=np.log1p(apps_df["Installs"])
apps_df["Log_Reviews"]=np.log1p(apps_df["Reviews"])

In [ ]:
apps_df.columns

In [ ]:
def rating_group(rating):
  if rating >=4:
    return "Top rated"
  elif rating >=3:
    return "Above average"
  if rating >=2:
    return "Average"
  else:
    return "Below average"


In [ ]:
apps_df["Rating_Group"]=apps_df["Rating"].apply(rating_group)

In [ ]:
apps_df["Rating_Group"]

In [ ]:
apps_df["Revenue"]=apps_df["Price"]*apps_df["Installs"]

In [ ]:
apps_df.columns


In [ ]:
sia=SentimentIntensityAnalyzer()

In [ ]:
review="this app is amazing ! i love the features"
s=sia.polarity_scores(review)
print(s)

In [ ]:
review="this app is  wrost ! i hate   the features"
s=sia.polarity_scores(review)
print(s)

In [ ]:
review="this app is average"
s=sia.polarity_scores(review)
print(s)

In [ ]:
reviews_df["Sentiment_Score"]=reviews_df["Translated_Review"].apply(lambda x:sia.polarity_scores(str(x))["compound"])

In [ ]:
#
sentiment_counts = reviews_df['Sentiment_Score'].value_counts()

def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

reviews_df["Sentiment_Class"] = reviews_df["Sentiment_Score"].apply(
    classify_sentiment
)

reviews_df["Sentiment_Class"].value_counts()

In [ ]:
reviews_df.head()

In [ ]:
apps_df['Last Updated']=pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [ ]:
apps_df['Year']=apps_df['Last Updated'].dt.year

In [ ]:
apps_df.head()

In [ ]:
import os

html_files_path = "Results"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [ ]:
plot_containers="/content/Results"

EDA

In [ ]:
import plotly.io as pio


def save_plot_as_html(fig, filename, insight):
  global plot_containers
  filepath = os.path.join(html_files_path, filename)
  html_content = pio.to_html(fig, full_html=False, include_plotlyjs="inline")

  # Append the plot and its insight to plot_containers
  plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
  fig.write_html(filepath, full_html=False, include_plotlyjs="inline")

In [ ]:
import plotly as px
plot_width=400
plot_height=300
plot_bg_color="black"
text_color="white"
title_font={'size':16}
axis_font={'size':12}

In [ ]:
import plotly.express as px

# Prepare data
category_counts = apps_df["Category"].value_counts().nlargest(10)

# Create bar plot
fig1 = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={"x": "Category", "y": "Count"},
    title="Top Categories on Play Store",
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,
    height=300,
)

# Update layout aesthetics
fig1.update_layout(
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color="white",
    title_font={"size": 16},
    xaxis=dict(title_font={"size": 12}),
    yaxis=dict(title_font={"size": 12}),
    margin=dict(l=10, r=10, t=30, b=10),
)

# Save plot output
save_plot_as_html(
    fig1,
    "Category Graph 1.html",
    "The top categories on the Play Store are dominated by tools, entertainment, and productivity apps",
)
fig1.show()

In [ ]:
type_counts= apps_df["Type"].value_counts()
fig2=px.pie(
    values= type_counts.values,
    names=type_counts.index,
    title="App Type Distribution",
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=400,
    height=300

)

fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))

save_plot_as_html(fig2,"Type Graph 2.html","Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in app purchases")
fig2.show()

In [ ]:
fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=400,
    height=300
)
fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
fig3.show()
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users")

In [ ]:
sentiment_counts=reviews_df['Sentiment_Score'].value_counts()
fig4=px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,
    height=300
)
fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
fig4.show()
save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments")

In [ ]:
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=400,
    height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig5.show()
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig5,"Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal and daily usage")

In [ ]:
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig6.show()
save_plot_as_html(fig6, "Updates Graph 6.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

In [ ]:
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,
    height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig7.show()
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [ ]:
genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=400,
    height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig8.show()
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))

save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games")

In [ ]:
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig9.show()
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [ ]:
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
fig10.show()
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [ ]:
plot_containers_split=plot_containers.split('</dev>')

In [ ]:
plot_containers_split

In [ ]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [ ]:

if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

Dashboard 1

In [ ]:
dashboard_html = """
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>Google Play Store Review Analytics</title>

    <style>

        * {{
            box-sizing: border-box;
        }}

        body {{
            font-family: Arial, sans-serif;
            background-color: #1e1e1e;
            color: white;
            margin: 0;
            padding: 0;
        }}

        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 20px;
            padding: 20px;
            background-color: #303030;
            border-bottom: 2px solid #555;
        }}

        .header-logo {{
            width: 60px;
            height: 60px;
            object-fit: contain;
        }}

        .header-text {{
            text-align: center;
        }}

        .header-text h1 {{
            margin: 0;
            font-size: 28px;
        }}

        .header-text p {{
            margin: 5px 0 0;
            color: #aaa;
            font-size: 14px;
        }}

        .dsu-brand {{
            text-align: center;
            padding: 12px;
            background-color: #252525;
            color: #bbb;
            font-size: 14px;
        }}

        .dsu-brand strong {{
            color: white;
        }}

        .container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            padding: 20px;
        }}

        .plot-container {{
            border: 2px solid #555;
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
            background-color: #292929;
            border-radius: 10px;
            transition: 0.3s;
        }}

        .plot-container:hover {{
            transform: scale(1.02);
            border-color: #888;
            box-shadow: 0 5px 20px rgba(0,0,0,0.5);
        }}

        .plot-container img {{
            width: 100%;
            height: 100%;
            object-fit: contain;
        }}

        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.8);
            padding: 8px;
            border-radius: 5px;
            color: white;
            font-size: 12px;
        }}

        .plot-container:hover .insights {{
            display: block;
        }}

        footer {{
            text-align: center;
            padding: 20px;
            background-color: #181818;
            color: #888;
            font-size: 13px;
        }}

    </style>

    <script>

        function openPlot(filename) {{
            window.open(filename, '_blank');
        }}

    </script>

</head>

<body>

    <div class="header">

        <img
            class="header-logo"
            src="https://www.gstatic.com/images/branding/product/2x/googleg_48dp.png"
            alt="Google Logo"
        >

        <div class="header-text">

            <h1>
                Google Play Store Review Analytics
            </h1>

            <p>
                Data Analytics & Visualization Dashboard
            </p>

        </div>

        <img
            class="header-logo"
            src="https://upload.wikimedia.org/wikipedia/commons/7/78/Google_Play_Store_badge_EN.svg"
            alt="Google Play Store"
        >

    </div>


    <div class="dsu-brand">

        <strong>DSU AI & Robotics</strong>

        &nbsp; | &nbsp;

        Dayananda Sagar University

        &nbsp; | &nbsp;

        Prepared by <strong>Yaseen Akthar Maniyar</strong>

    </div>


    <div class="container">

        {plots}

    </div>


    <footer>

        Google Play Store Review Analytics

        <br>

        DSU AI & Robotics

        <br>

        © 2026 Yaseen Akthar Maniyar

    </footer>

</body>

</html>
"""

In [ ]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [ ]:
import requests

google_logo_url = "https://www.gstatic.com/images/branding/product/2x/googleg_48dp.png"

r = requests.get(google_logo_url)

with open("google_logo.png", "wb") as f:
    f.write(r.content)

In [ ]:
dashboard_path=os.path.join(html_files_path,"web page.html")


In [ ]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)


In [ ]:
from IPython.display import HTML, display
import os

# Path to your HTML file
dashboard_path = "/content/Results/web page.html"  # replace with your actual file path

# Read and render the HTML file directly in the notebook cell output
if os.path.exists(dashboard_path):
    with open(dashboard_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    display(HTML(html_content))
else:
    print(f"File not found at: {dashboard_path}")

Advanced EDA

In [ ]:
px.scatter(
    apps_df,
    x="Reviews",
    y="Rating",
    log_x=True,
    title="Reviews vs App Rating"
)

In [ ]:
apps_df[["Reviews", "Rating"]].corr()

In [ ]:
px.scatter(
    apps_df,
    x="Reviews",
    y="Installs",
    log_x=True,
    log_y=True,
    title="Reviews vs Installs"
)

In [ ]:
paid_apps = apps_df[apps_df["Type"] == "Paid"]

px.scatter(
    paid_apps,
    x="Price",
    y="Rating",
    title="Price vs Rating"
)

In [ ]:
category_analysis = apps_df.groupby("Category").agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Total_Installs=("Installs", "sum"),
    Total_Revenue=("Revenue", "sum")
).reset_index()

In [ ]:
category_analysis.sort_values(
    "Total_Installs",
    ascending=False
)

In [ ]:
apps_df["Success_Score"] = (
    apps_df["Rating"] * np.log1p(apps_df["Reviews"])
)

In [ ]:
apps_df["Success_Score"]

In [ ]:


import numpy as np
import pandas as pd

apps_df["Success_Score"] = (
    apps_df["Rating"] * np.log1p(apps_df["Reviews"])
)

print("=" * 80)
print("1. SUCCESS SCORE CREATED")
print("=" * 80)

print(apps_df[[
    "App",
    "Rating",
    "Reviews",
    "Success_Score"
]].head(10))


top_successful_apps = apps_df.sort_values(
    by="Success_Score",
    ascending=False
)[[
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Type",
    "Success_Score"
]].head(20)

print("\n")
print("=" * 80)
print("2. TOP 20 SUCCESSFUL APPS")
print("=" * 80)

print(top_successful_apps.to_string(index=False))


print("\n")
print("=" * 80)
print("3. SUCCESS SCORE BY RATING")
print("=" * 80)

rating_analysis = apps_df.groupby("Rating").agg(
    App_Count=("App", "count"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(rating_analysis.to_string(index=False))


print("\n")
print("=" * 80)
print("4. SUCCESS SCORE BY REVIEWS")
print("=" * 80)


apps_df["Review_Group"] = pd.cut(
    apps_df["Reviews"],
    bins=[-1, 100, 1000, 10000, 100000, 1000000, np.inf],
    labels=[
        "0-100",
        "101-1K",
        "1K-10K",
        "10K-100K",
        "100K-1M",
        "1M+"
    ]
)

reviews_analysis = apps_df.groupby(
    "Review_Group",
    observed=True
).agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(reviews_analysis.to_string(index=False))



print("\n")
print("=" * 80)
print("5. SUCCESS SCORE BY INSTALLS")
print("=" * 80)

apps_df["Install_Group"] = pd.cut(
    apps_df["Installs"],
    bins=[-1, 1000, 10000, 100000, 1000000, 10000000,
          100000000, 1000000000, np.inf],
    labels=[
        "0-1K",
        "1K-10K",
        "10K-100K",
        "100K-1M",
        "1M-10M",
        "10M-100M",
        "100M-1B",
        "1B+"
    ]
)

install_analysis = apps_df.groupby(
    "Install_Group",
    observed=True
).agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(install_analysis.to_string(index=False))

print("\n")
print("=" * 80)
print("6. SUCCESS SCORE BY PRICE")
print("=" * 80)

apps_df["Price_Group"] = pd.cut(
    apps_df["Price"],
    bins=[-0.01, 0, 5, 10, 20, 50, np.inf],
    labels=[
        "Free",
        "$0-$5",
        "$5-$10",
        "$10-$20",
        "$20-$50",
        "$50+"
    ]
)

price_analysis = apps_df.groupby(
    "Price_Group",
    observed=True
).agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(price_analysis.to_string(index=False))

print("\n")
print("=" * 80)
print("7. CATEGORY-WISE SUCCESS ANALYSIS")
print("=" * 80)

category_analysis = apps_df.groupby("Category").agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Total_Installs=("Installs", "sum"),
    Avg_Price=("Price", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

category_analysis = category_analysis.sort_values(
    "Avg_Success_Score",
    ascending=False
)

print(category_analysis.to_string(index=False))



print("\n")
print("=" * 80)
print("8. APP SIZE VS SUCCESS")
print("=" * 80)

apps_df["Size_Group"] = pd.cut(
    apps_df["Size"],
    bins=[-np.inf, 10, 25, 50, 100, np.inf],
    labels=[
        "<10 MB",
        "10-25 MB",
        "25-50 MB",
        "50-100 MB",
        "100+ MB"
    ]
)

size_analysis = apps_df.groupby(
    "Size_Group",
    observed=True
).agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(size_analysis.to_string(index=False))



print("\n")
print("=" * 80)
print("9. UPDATE YEAR VS SUCCESS")
print("=" * 80)



if "Year" in apps_df.columns:

    apps_df["Update_Year"] = apps_df["Year"]

elif "Last Updated" in apps_df.columns:

    apps_df["Update_Year"] = pd.to_datetime(
        apps_df["Last Updated"],
        errors="coerce"
    ).dt.year

elif "Last_Updated" in apps_df.columns:

    apps_df["Update_Year"] = pd.to_datetime(
        apps_df["Last_Updated"],
        errors="coerce"
    ).dt.year

else:

    apps_df["Update_Year"] = np.nan


year_analysis = apps_df.groupby(
    "Update_Year",
    dropna=True
).agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

year_analysis = year_analysis.sort_values(
    "Update_Year"
)

print(year_analysis.to_string(index=False))



print("\n")
print("=" * 80)
print("10. FREE VS PAID SUCCESS ANALYSIS")
print("=" * 80)

type_analysis = apps_df.groupby("Type").agg(
    App_Count=("App", "count"),
    Avg_Rating=("Rating", "mean"),
    Avg_Reviews=("Reviews", "mean"),
    Avg_Installs=("Installs", "mean"),
    Avg_Price=("Price", "mean"),
    Avg_Success_Score=("Success_Score", "mean")
).reset_index()

print(type_analysis.to_string(index=False))



print("\n")
print("=" * 80)
print("11. CORRELATION WITH SUCCESS SCORE")
print("=" * 80)

correlation_columns = [
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Size",
    "Success_Score"
]

correlation_columns = [
    col for col in correlation_columns
    if col in apps_df.columns
]

correlation_matrix = apps_df[
    correlation_columns
].corr()

print(correlation_matrix.round(3).to_string())


print("\n")
print("Correlation of variables with Success Score:")
print("=" * 80)

success_correlation = (
    correlation_matrix["Success_Score"]
    .sort_values(ascending=False)
)

print(success_correlation.round(3).to_string())




print("\n")
print("=" * 80)
print("12. FINAL APP SUCCESS DATA")
print("=" * 80)

final_success_table = apps_df[[
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Size",
    "Update_Year",
    "Type",
    "Success_Score"
]].sort_values(
    "Success_Score",
    ascending=False
)

print(final_success_table.head(30).to_string(index=False))



print("\n")
print("=" * 80)
print("13. SUCCESS SCORE SUMMARY")
print("=" * 80)

print(
    apps_df["Success_Score"].describe().round(3)
)




print("\n")
print("=" * 80)
print("14. TOP 10 CATEGORIES BY AVERAGE SUCCESS SCORE")
print("=" * 80)

top_categories = category_analysis[
    [
        "Category",
        "App_Count",
        "Avg_Rating",
        "Avg_Reviews",
        "Avg_Installs",
        "Avg_Success_Score"
    ]
].head(10)

print(top_categories.to_string(index=False))




print("\n")
print("=" * 80)
print("SUCCESS ANALYSIS COMPLETED")
print("=" * 80)

print(f"Total Apps Analyzed: {len(apps_df):,}")
print(
    f"Average Success Score: "
    f"{apps_df['Success_Score'].mean():.3f}"
)
print(
    f"Highest Success Score: "
    f"{apps_df['Success_Score'].max():.3f}"
)

print("\nTop Successful App:")
top_app = apps_df.loc[
    apps_df["Success_Score"].idxmax()
]

print(top_app[[
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Type",
    "Success_Score"
]].to_string())

Correlation heatmap


In [ ]:
numeric_cols = [
    "Rating",
    "Reviews",
    "Size",
    "Installs",
    "Price",
    "Log_Installs",
    "Log_Reviews",
    "Revenue"
]

corr = apps_df[numeric_cols].corr()

In [ ]:
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from scipy.stats import ttest_ind, sem, t


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("=" * 80)
print("GOOGLE PLAY STORE - STATISTICAL & PREDICTIVE ANALYTICS")
print("=" * 80)




print("\n" + "=" * 80)
print("1. DATA PREPARATION")
print("=" * 80)


df = apps_df.copy()



numeric_columns = [
    "Rating",
    "Reviews",
    "Size",
    "Installs",
    "Price"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")




if "Year" not in df.columns:

    if "Last Updated" in df.columns:

        df["Last Updated"] = pd.to_datetime(
            df["Last Updated"],
            errors="coerce"
        )

        df["Year"] = df["Last Updated"].dt.year

    else:
        df["Year"] = np.nan




if "Rating_Group" not in df.columns:

    def rating_group(rating):

        if pd.isna(rating):
            return np.nan

        if rating >= 4:
            return "Top rated"

        elif rating >= 3:
            return "Above average"

        elif rating >= 2:
            return "Average"

        else:
            return "Below average"

    df["Rating_Group"] = df["Rating"].apply(rating_group)


print("\nDataset shape:")
print(df.shape)

print("\nColumns used in analysis:")
print(df.columns.tolist())




print("\n" + "=" * 80)
print("2. STATISTICAL ANALYSIS - PAID VS FREE")
print("=" * 80)

print("""
H0 (Null Hypothesis):
There is NO significant difference in ratings
between paid and free apps.

H1 (Alternative Hypothesis):
There IS a significant difference in ratings
between paid and free apps.

Statistical Test:
Independent Two-Sample Welch's t-test
Significance Level (alpha): 0.05
""")




free_ratings = df.loc[
    df["Type"].astype(str).str.strip().str.lower() == "free",
    "Rating"
].dropna()

paid_ratings = df.loc[
    df["Type"].astype(str).str.strip().str.lower() == "paid",
    "Rating"
].dropna()


print("Number of Free Apps :", len(free_ratings))
print("Number of Paid Apps :", len(paid_ratings))



free_mean = free_ratings.mean()
paid_mean = paid_ratings.mean()

free_std = free_ratings.std()
paid_std = paid_ratings.std()


print("\nAverage Rating:")
print(f"Free Apps : {free_mean:.4f}")
print(f"Paid Apps : {paid_mean:.4f}")

print("\nStandard Deviation:")
print(f"Free Apps : {free_std:.4f}")
print(f"Paid Apps : {paid_std:.4f}")




t_statistic, p_value = ttest_ind(
    free_ratings,
    paid_ratings,
    equal_var=False
)


print("\nWelch's Independent Two-Sample t-test")
print("-" * 50)

print(f"T-statistic : {t_statistic:.4f}")
print(f"P-value     : {p_value:.6f}")




alpha = 0.05

print("\nStatistical Decision:")

if p_value < alpha:

    print("REJECT H0")
    print(
        "There is a statistically significant difference "
        "between ratings of paid and free apps."
    )

else:

    print("FAIL TO REJECT H0")
    print(
        "There is NOT enough statistical evidence to conclude "
        "that paid and free apps have different average ratings."
    )




print("\n" + "=" * 80)
print("3. 95% CONFIDENCE INTERVAL")
print("=" * 80)

mean_difference = paid_mean - free_mean


se_difference = np.sqrt(
    (paid_std ** 2 / len(paid_ratings)) +
    (free_std ** 2 / len(free_ratings))
)


numerator = (
    (paid_std ** 2 / len(paid_ratings)) +
    (free_std ** 2 / len(free_ratings))
) ** 2

denominator = (
    ((paid_std ** 2 / len(paid_ratings)) ** 2) /
    (len(paid_ratings) - 1)
) + (
    ((free_std ** 2 / len(free_ratings)) ** 2) /
    (len(free_ratings) - 1)
)

degrees_freedom = numerator / denominator

critical_value = t.ppf(
    0.975,
    degrees_freedom
)

margin_error = critical_value * se_difference

ci_lower = mean_difference - margin_error
ci_upper = mean_difference + margin_error


print(f"Mean Difference (Paid - Free): {mean_difference:.4f}")
print(f"95% CI Lower Bound           : {ci_lower:.4f}")
print(f"95% CI Upper Bound           : {ci_upper:.4f}")




print("\n" + "=" * 80)
print("4. EFFECT SIZE - COHEN'S d")
print("=" * 80)


pooled_std = np.sqrt(
    (
        ((len(paid_ratings) - 1) * paid_std ** 2) +
        ((len(free_ratings) - 1) * free_std ** 2)
    )
    /
    (
        len(paid_ratings) +
        len(free_ratings) - 2
    )
)

cohens_d = mean_difference / pooled_std

print(f"Cohen's d : {cohens_d:.4f}")


if abs(cohens_d) < 0.2:
    effect_interpretation = "Very small effect"

elif abs(cohens_d) < 0.5:
    effect_interpretation = "Small effect"

elif abs(cohens_d) < 0.8:
    effect_interpretation = "Medium effect"

else:
    effect_interpretation = "Large effect"


print("Interpretation:", effect_interpretation)



print("\n" + "=" * 80)
print("5. PAID VS FREE RATING DISTRIBUTION")
print("=" * 80)

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df[
        df["Type"].astype(str).str.strip().str.lower().isin(
            ["free", "paid"]
        )
    ],
    x="Type",
    y="Rating"
)

plt.title("Rating Distribution: Paid vs Free Apps")
plt.xlabel("App Type")
plt.ylabel("Rating")
plt.grid(axis="y", alpha=0.3)

plt.show()




print("\n" + "=" * 80)
print("6. PREDICTIVE ANALYTICS")
print("=" * 80)

print("""
Prediction Problem:

INPUT FEATURES
    Category
    Reviews
    Size
    Installs
    Price
    Type
    Content Rating
    Genres
    Year

             ↓

Random Forest Classifier

             ↓

TARGET
    Rating_Group
""")





feature_columns = [
    "Category",
    "Reviews",
    "Size",
    "Installs",
    "Price",
    "Type",
    "Content Rating",
    "Genres",
    "Year"
]



feature_columns = [
    col for col in feature_columns
    if col in df.columns
]


target_column = "Rating_Group"


print("Features:")
print(feature_columns)

print("\nTarget:")
print(target_column)




ml_df = df[
    feature_columns + [target_column]
].dropna(
    subset=[target_column]
).copy()


X = ml_df[feature_columns]

y = ml_df[target_column]


print("\nML Dataset Shape:")
print(ml_df.shape)




print("\n" + "=" * 80)
print("7. TARGET DISTRIBUTION")
print("=" * 80)

target_distribution = y.value_counts()

print(target_distribution)

print("\nTarget Percentage:")
print(
    (y.value_counts(normalize=True) * 100)
    .round(2)
)




numerical_features = [
    col for col in feature_columns
    if col in [
        "Reviews",
        "Size",
        "Installs",
        "Price",
        "Year"
    ]
]

categorical_features = [
    col for col in feature_columns
    if col not in numerical_features
]


print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)




# Numerical preprocessing
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))




model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)




rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            model
        )
    ]
)




print("\n" + "=" * 80)
print("8. TRAINING RANDOM FOREST")
print("=" * 80)

rf_pipeline.fit(
    X_train,
    y_train
)

print("Model training completed successfully.")




y_pred = rf_pipeline.predict(X_test)




print("\n" + "=" * 80)
print("9. MODEL EVALUATION")
print("=" * 80)


accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)


print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")




print("\n" + "=" * 80)
print("10. CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)




print("\n" + "=" * 80)
print("11. CONFUSION MATRIX")
print("=" * 80)

labels = sorted(
    y.unique()
)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)


print("\nConfusion Matrix:")
print(cm)


plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.title("Random Forest - Rating Group Confusion Matrix")
plt.xlabel("Predicted Rating Group")
plt.ylabel("Actual Rating Group")

plt.tight_layout()
plt.show()


print("\n" + "=" * 80)
print("12. FEATURE IMPORTANCE")
print("=" * 80)


# Get trained Random Forest
trained_model = rf_pipeline.named_steps["classifier"]

# Get preprocessing object
trained_preprocessor = rf_pipeline.named_steps["preprocessor"]


# Get transformed feature names
feature_names = trained_preprocessor.get_feature_names_out()


# Get importance values
importance_values = trained_model.feature_importances_


feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})


# Remove preprocessing prefixes
feature_importance["Feature"] = (
    feature_importance["Feature"]
    .str.replace(
        "num__",
        "",
        regex=False
    )
    .str.replace(
        "cat__",
        "",
        regex=False
    )
)


# Sort
feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nTop 20 Important Features:")
print(
    feature_importance.head(20).to_string(index=False)
)




top_features = feature_importance.head(15).sort_values(
    "Importance"
)


plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title(
    "Top 15 Features Predicting App Rating Group"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()




print("\n" + "=" * 80)
print("13. FINAL ANALYTICAL SUMMARY")
print("=" * 80)


print("\nSTATISTICAL ANALYSIS")
print("-" * 50)

print(f"Free App Average Rating : {free_mean:.4f}")
print(f"Paid App Average Rating : {paid_mean:.4f}")
print(f"T-statistic             : {t_statistic:.4f}")
print(f"P-value                 : {p_value:.6f}")
print(f"Cohen's d               : {cohens_d:.4f}")
print(f"Effect                  : {effect_interpretation}")


if p_value < alpha:

    print(
        "\nConclusion:"
        "\nThere is statistically significant evidence "
        "of a difference in average ratings between "
        "paid and free apps."
    )

else:

    print(
        "\nConclusion:"
        "\nThere is insufficient statistical evidence "
        "to conclude that paid and free apps have "
        "different average ratings."
    )


print("\nMACHINE LEARNING")
print("-" * 50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


print("\nMost Important Predictive Features:")

for i, row in feature_importance.head(10).iterrows():

    print(
        f"{i+1}. {row['Feature']} "
        f"-> {row['Importance']:.4f}"
    )


print("\n" + "=" * 80)
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:


import pandas as pd
import numpy as np
import json
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.utils import PlotlyJSONEncoder
from IPython.display import display, HTML, IFrame




df = apps_df.copy()



for col in ["Rating", "Reviews", "Size", "Installs", "Price"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")




if "Year" not in df.columns:

    if "Last Updated" in df.columns:

        df["Last Updated"] = pd.to_datetime(
            df["Last Updated"],
            errors="coerce"
        )

        df["Year"] = df["Last Updated"].dt.year

    else:
        df["Year"] = np.nan




if "Rating_Group" not in df.columns:

    def rating_group(rating):

        if pd.isna(rating):
            return np.nan

        if rating >= 4:
            return "Top rated"

        elif rating >= 3:
            return "Above average"

        elif rating >= 2:
            return "Average"

        else:
            return "Below average"

    df["Rating_Group"] = df["Rating"].apply(rating_group)



if "Revenue" not in df.columns:

    df["Revenue"] = (
        df["Price"].fillna(0) *
        df["Installs"].fillna(0)
    )



if "reviews_df" in globals():

    try:

        sentiment_df = reviews_df.copy()

        if "Sentiment_Score" in sentiment_df.columns:

            sentiment_df["Sentiment_Score"] = pd.to_numeric(
                sentiment_df["Sentiment_Score"],
                errors="coerce"
            )

            # Convert sentiment score to class
            def sentiment_class(score):

                if pd.isna(score):
                    return "Unknown"

                if score >= 0.05:
                    return "Positive"

                elif score <= -0.05:
                    return "Negative"

                else:
                    return "Neutral"

            sentiment_df["Sentiment"] = (
                sentiment_df["Sentiment_Score"]
                .apply(sentiment_class)
            )

            # Aggregate sentiment per app
            app_sentiment = (
                sentiment_df
                .groupby("App")
                .agg(
                    Sentiment_Score=("Sentiment_Score", "mean"),
                    Sentiment=("Sentiment", lambda x:
                               x.value_counts().index[0]
                               if len(x) > 0 else "Unknown")
                )
                .reset_index()
            )

            # Merge
            df = df.merge(
                app_sentiment,
                on="App",
                how="left"
            )

        else:

            df["Sentiment_Score"] = np.nan
            df["Sentiment"] = "Unknown"

    except Exception as e:

        print("Sentiment processing skipped:", e)

        df["Sentiment_Score"] = np.nan
        df["Sentiment"] = "Unknown"

else:

    df["Sentiment_Score"] = np.nan
    df["Sentiment"] = "Unknown"




dashboard_df = df.copy()

dashboard_df = dashboard_df[
    dashboard_df["App"].notna()
].copy()



categorical_cols = [
    "Category",
    "Type",
    "Content Rating",
    "Genres",
    "Rating_Group",
    "Sentiment"
]

for col in categorical_cols:

    if col in dashboard_df.columns:

        dashboard_df[col] = (
            dashboard_df[col]
            .fillna("Unknown")
            .astype(str)
        )


# Fill numerical values
dashboard_df["Rating"] = dashboard_df["Rating"].fillna(
    dashboard_df["Rating"].median()
)

dashboard_df["Reviews"] = dashboard_df["Reviews"].fillna(0)

dashboard_df["Installs"] = dashboard_df["Installs"].fillna(0)

dashboard_df["Price"] = dashboard_df["Price"].fillna(0)

dashboard_df["Revenue"] = dashboard_df["Revenue"].fillna(0)

dashboard_df["Year"] = dashboard_df["Year"].fillna(0)




dashboard_df = dashboard_df[
    (dashboard_df["Rating"] >= 0) &
    (dashboard_df["Rating"] <= 5)
].copy()




dashboard_columns = [
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Size",
    "Installs",
    "Type",
    "Price",
    "Revenue",
    "Content Rating",
    "Genres",
    "Year",
    "Rating_Group",
    "Sentiment",
    "Sentiment_Score"
]

dashboard_columns = [
    c for c in dashboard_columns
    if c in dashboard_df.columns
]

js_data = dashboard_df[
    dashboard_columns
].to_dict(
    orient="records"
)




feature_importance_data = []

# If feature_importance was created in the previous ML section,
# use it automatically.

if "feature_importance" in globals():

    try:

        fi = feature_importance.copy()

        fi = fi.head(15)

        for _, row in fi.iterrows():

            feature_importance_data.append({
                "Feature": str(row["Feature"]),
                "Importance": float(row["Importance"])
            })

    except Exception as e:

        print("Feature importance could not be loaded:", e)


# If unavailable, create a placeholder.
if len(feature_importance_data) == 0:

    feature_importance_data = [
        {
            "Feature": "Run ML model first",
            "Importance": 1
        }
    ]



categories = sorted(
    dashboard_df["Category"]
    .dropna()
    .unique()
    .tolist()
)

types = sorted(
    dashboard_df["Type"]
    .dropna()
    .unique()
    .tolist()
)

content_ratings = sorted(
    dashboard_df["Content Rating"]
    .dropna()
    .unique()
    .tolist()
)

years = sorted([
    int(y)
    for y in dashboard_df["Year"].unique()
    if y > 0
])




data_json = json.dumps(
    js_data,
    cls=PlotlyJSONEncoder
)

feature_json = json.dumps(
    feature_importance_data,
    cls=PlotlyJSONEncoder
)


categories_json = json.dumps(categories)

types_json = json.dumps(types)

content_json = json.dumps(content_ratings)

years_json = json.dumps(years)




html = f"""

<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>
Google Play Store Analytics Dashboard
</title>


<!-- Plotly -->
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>


<style>

* {{
    box-sizing: border-box;
}}

body {{

    margin: 0;

    font-family:
        Arial,
        Helvetica,
        sans-serif;

    background:
        linear-gradient(
            135deg,
            #0f172a,
            #111827
        );

    color: white;

}}


.container {{

    width: 95%;

    max-width: 1500px;

    margin: auto;

    padding: 25px 0 50px;

}}


/* ==========================================================
HEADER
========================================================== */

.header {{

    background:
        linear-gradient(
            135deg,
            #1e293b,
            #0f172a
        );

    border-radius: 20px;

    padding: 30px;

    text-align: center;

    margin-bottom: 20px;

    border:
        1px solid
        rgba(255,255,255,0.1);

    box-shadow:
        0 10px 30px
        rgba(0,0,0,0.3);

}}

.header h1 {{

    margin: 0;

    font-size: 32px;

}}

.header p {{

    margin-top: 8px;

    color: #cbd5e1;

    font-size: 15px;

}}


/* ==========================================================
FILTER PANEL
========================================================== */

.filters {{

    display: grid;

    grid-template-columns:
        repeat(
            auto-fit,
            minmax(180px,1fr)
        );

    gap: 15px;

    background:
        #1e293b;

    padding: 20px;

    border-radius: 18px;

    margin-bottom: 20px;

}}

.filter-box label {{

    display: block;

    margin-bottom: 7px;

    font-size: 13px;

    color: #cbd5e1;

}}

select,
input {{

    width: 100%;

    padding: 10px;

    border-radius: 8px;

    border: none;

    background: #334155;

    color: white;

    outline: none;

}}


.range-row {{

    display: flex;

    gap: 8px;

}}


.range-row input {{

    width: 50%;

}}


/* ==========================================================
KPI CARDS
========================================================== */

.kpis {{

    display: grid;

    grid-template-columns:
        repeat(
            auto-fit,
            minmax(200px,1fr)
        );

    gap: 15px;

    margin-bottom: 20px;

}}


.kpi {{

    background:
        #1e293b;

    padding: 22px;

    border-radius: 18px;

    text-align: center;

    border:
        1px solid
        rgba(255,255,255,0.08);

}}


.kpi-title {{

    color: #94a3b8;

    font-size: 13px;

}}


.kpi-value {{

    font-size: 28px;

    font-weight: bold;

    margin-top: 8px;

    color: #38bdf8;

}}


/* ==========================================================
SECTIONS
========================================================== */

.section-title {{

    font-size: 22px;

    margin:
        30px 0 12px;

}}


.grid-2 {{

    display: grid;

    grid-template-columns:
        repeat(
            auto-fit,
            minmax(400px,1fr)
        );

    gap: 18px;

}}


.chart-card {{

    background: #1e293b;

    border-radius: 18px;

    padding: 10px;

    min-height: 380px;

}}


/* ==========================================================
TOP APPS TABLE
========================================================== */

.table-card {{

    background: #1e293b;

    border-radius: 18px;

    padding: 20px;

    overflow-x: auto;

}}

table {{

    width: 100%;

    border-collapse: collapse;

}}

th,
td {{

    padding: 11px;

    text-align: left;

    border-bottom:
        1px solid
        #334155;

}}

th {{

    color: #38bdf8;

}}

td {{

    color: #e2e8f0;

}}


/* ==========================================================
FOOTER
========================================================== */

.footer {{

    margin-top: 40px;

    text-align: center;

    padding: 20px;

    color: #94a3b8;

    border-top:
        1px solid
        #334155;

}}


/* ==========================================================
RESPONSIVE
========================================================== */

@media(max-width:700px) {{

    .container {{

        width: 96%;

    }}

    .header h1 {{

        font-size: 24px;

    }}

}}

</style>

</head>


<body>


<div class="container">


<!-- ======================================================
HEADER
====================================================== -->

<div class="header">

<h1>
GOOGLE PLAY STORE ANALYTICS
</h1>

<p>
Interactive Business Intelligence Dashboard
</p>

<p>
Yaseen Akthar M • AI & Robotics • Dayananda Sagar University
</p>

</div>


<!-- ======================================================
FILTERS
====================================================== -->

<div class="filters">


<div class="filter-box">

<label>
Category
</label>

<select id="categoryFilter">

<option value="All">
All Categories
</option>

</select>

</div>


<div class="filter-box">

<label>
App Type
</label>

<select id="typeFilter">

<option value="All">
All Types
</option>

</select>

</div>


<div class="filter-box">

<label>
Content Rating
</label>

<select id="contentFilter">

<option value="All">
All
</option>

</select>

</div>


<div class="filter-box">

<label>
Year
</label>

<select id="yearFilter">

<option value="All">
All Years
</option>

</select>

</div>


<div class="filter-box">

<label>
Minimum Installs
</label>

<select id="installFilter">

<option value="0">
All
</option>

<option value="1000">
1K+
</option>

<option value="10000">
10K+
</option>

<option value="100000">
100K+
</option>

<option value="1000000">
1M+
</option>

<option value="10000000">
10M+
</option>

<option value="100000000">
100M+
</option>

</select>

</div>


<div class="filter-box">

<label>
Rating Range
</label>

<div class="range-row">

<input
type="number"
id="ratingMin"
min="0"
max="5"
step="0.1"
value="0"
>

<input
type="number"
id="ratingMax"
min="0"
max="5"
step="0.1"
value="5"
>

</div>

</div>


</div>


<!-- ======================================================
KPI CARDS
====================================================== -->

<div class="kpis">


<div class="kpi">

<div class="kpi-title">
TOTAL APPS
</div>

<div
class="kpi-value"
id="totalApps"
>
0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
AVERAGE RATING
</div>

<div
class="kpi-value"
id="avgRating"
>
0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
TOTAL INSTALLS
</div>

<div
class="kpi-value"
id="totalInstalls"
>
0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
EST. REVENUE
</div>

<div
class="kpi-value"
id="totalRevenue"
>
₹0
</div>

</div>


</div>


<!-- ======================================================
CATEGORY PERFORMANCE
====================================================== -->

<div class="section-title">

📊 Category Performance

</div>


<div class="grid-2">


<div class="chart-card">

<div id="categoryInstalls"></div>

</div>


<div class="chart-card">

<div id="categoryRevenue"></div>

</div>


</div>


<!-- ======================================================
RATING ANALYSIS
====================================================== -->

<div class="section-title">

⭐ Rating Analysis

</div>


<div class="grid-2">


<div class="chart-card">

<div id="ratingDistribution"></div>

</div>


<div class="chart-card">

<div id="reviewsRating"></div>

</div>


</div>


<!-- ======================================================
SENTIMENT
====================================================== -->

<div class="section-title">

💬 User Sentiment

</div>


<div class="grid-2">


<div class="chart-card">

<div id="sentimentChart"></div>

</div>


<div class="chart-card">

<div id="ratingGroupChart"></div>

</div>


</div>


<!-- ======================================================
SUCCESS ANALYSIS
====================================================== -->

<div class="section-title">

🏆 App Success Analysis

</div>


<div class="table-card">

<table>

<thead>

<tr>

<th>
Rank
</th>

<th>
App
</th>

<th>
Category
</th>

<th>
Rating
</th>

<th>
Reviews
</th>

<th>
Installs
</th>

<th>
Type
</th>

<th>
Success Score
</th>

</tr>

</thead>

<tbody id="topAppsTable">

</tbody>

</table>

</div>


<!-- ======================================================
PREDICTIVE ANALYTICS
====================================================== -->

<div class="section-title">

🤖 Predictive Analytics

</div>


<div class="chart-card">

<div id="featureImportance"></div>

</div>


<!-- ======================================================
FOOTER
====================================================== -->

<div class="footer">

DSU AI & Robotics |
Dayananda Sagar University

<br><br>

Google Play Store Data Analytics Project

<br><br>

Estimated revenue is a proxy calculated using
Price × Installs.

</div>


</div>


<!-- ======================================================
JAVASCRIPT
====================================================== -->

<script>


// ==========================================================
// LOAD DATA
// ==========================================================

const originalData =
{data_json};


const featureImportance =
{feature_json};


const categories =
{categories_json};


const types =
{types_json};


const contentRatings =
{content_json};


const years =
{years_json};


// ==========================================================
// POPULATE FILTERS
// ==========================================================

function populateSelect(
    elementId,
    values
) {{

    const select =
        document.getElementById(
            elementId
        );

    values.forEach(
        value => {{

            const option =
                document.createElement(
                    "option"
                );

            option.value = value;

            option.textContent = value;

            select.appendChild(
                option
            );

        }}
    );

}}


populateSelect(
    "categoryFilter",
    categories
);

populateSelect(
    "typeFilter",
    types
);

populateSelect(
    "contentFilter",
    contentRatings
);

populateSelect(
    "yearFilter",
    years
);


// ==========================================================
// FORMAT NUMBERS
// ==========================================================

function formatNumber(value) {{

    if (!isFinite(value))
        return "0";

    if (value >= 1000000000)
        return (
            (value / 1000000000)
            .toFixed(2) + "B"
        );

    if (value >= 1000000)
        return (
            (value / 1000000)
            .toFixed(2) + "M"
        );

    if (value >= 1000)
        return (
            (value / 1000)
            .toFixed(2) + "K"
        );

    return Math.round(value).toLocaleString();

}}


function formatCurrency(value) {{

    if (!isFinite(value))
        return "₹0";

    if (value >= 1000000000)
        return (
            "₹" +
            (value / 1000000000)
            .toFixed(2) +
            "B"
        );

    if (value >= 1000000)
        return (
            "₹" +
            (value / 1000000)
            .toFixed(2) +
            "M"
        );

    if (value >= 1000)
        return (
            "₹" +
            (value / 1000)
            .toFixed(2) +
            "K"
        );

    return (
        "₹" +
        Math.round(value)
        .toLocaleString()
    );

}}


// ==========================================================
// FILTER DATA
// ==========================================================

function getFilteredData() {{

    const category =
        document.getElementById(
            "categoryFilter"
        ).value;

    const type =
        document.getElementById(
            "typeFilter"
        ).value;

    const content =
        document.getElementById(
            "contentFilter"
        ).value;

    const year =
        document.getElementById(
            "yearFilter"
        ).value;

    const minInstalls =
        Number(
            document.getElementById(
                "installFilter"
            ).value
        );

    const minRating =
        Number(
            document.getElementById(
                "ratingMin"
            ).value
        );

    const maxRating =
        Number(
            document.getElementById(
                "ratingMax"
            ).value
        );


    return originalData.filter(
        app => {{

            const categoryOK =
                category === "All" ||
                app["Category"] === category;


            const typeOK =
                type === "All" ||
                app["Type"] === type;


            const contentOK =
                content === "All" ||
                app["Content Rating"] === content;


            const yearOK =
                year === "All" ||
                Number(app["Year"]) === Number(year);


            const installsOK =
                Number(app["Installs"] || 0)
                >= minInstalls;


            const ratingOK =
                Number(app["Rating"] || 0)
                >= minRating &&
                Number(app["Rating"] || 0)
                <= maxRating;


            return (
                categoryOK &&
                typeOK &&
                contentOK &&
                yearOK &&
                installsOK &&
                ratingOK
            );

        }}
    );

}}


// ==========================================================
// COMMON PLOT STYLE
// ==========================================================

const plotLayout = {{

    paper_bgcolor:
        "#1e293b",

    plot_bgcolor:
        "#1e293b",

    font: {{

        color: "#e2e8f0"

    }},

    margin: {{

        l: 60,
        r: 30,
        t: 60,
        b: 60

    }},

    title: {{

        font: {{

            size: 18

        }}

    }}

}};


// ==========================================================
// UPDATE DASHBOARD
// ==========================================================

function updateDashboard() {{

    const data =
        getFilteredData();


    // ========================================================
    // KPIs
    // ========================================================

    const totalApps =
        data.length;


    const avgRating =
        data.length > 0
        ?
        data.reduce(
            (sum, app) =>
                sum +
                Number(app.Rating || 0),
            0
        ) / data.length
        :
        0;


    const totalInstalls =
        data.reduce(
            (sum, app) =>
                sum +
                Number(app.Installs || 0),
            0
        );


    const totalRevenue =
        data.reduce(
            (sum, app) =>
                sum +
                Number(app.Revenue || 0),
            0
        );


    document.getElementById(
        "totalApps"
    ).textContent =
        formatNumber(totalApps);


    document.getElementById(
        "avgRating"
    ).textContent =
        avgRating.toFixed(2);


    document.getElementById(
        "totalInstalls"
    ).textContent =
        formatNumber(totalInstalls);


    document.getElementById(
        "totalRevenue"
    ).textContent =
        formatCurrency(totalRevenue);


    // ========================================================
    // CATEGORY INSTALLS
    // ========================================================

    const categoryMap = {{}};


    data.forEach(
        app => {{

            const category =
                app.Category || "Unknown";

            if (!categoryMap[category])
                categoryMap[category] = 0;

            categoryMap[category] +=
                Number(
                    app.Installs || 0
                );

        }}
    );


    const categoryData =
        Object.entries(
            categoryMap
        )
        .sort(
            (a,b) => b[1] - a[1]
        )
        .slice(0,10);


    Plotly.react(
        "categoryInstalls",
        [
            {{
                x:
                    categoryData.map(
                        x => x[1]
                    ),

                y:
                    categoryData.map(
                        x => x[0]
                    ),

                type: "bar",

                orientation: "h"

            }}
        ],
        {{
            ...plotLayout,

            title:
                "Top Categories by Installs",

            xaxis:
                {{
                    title:
                        "Installs"
                }},

            yaxis:
                {{
                    automargin: true
                }}

        }},
        {{responsive:true}}
    );


    // ========================================================
    // CATEGORY REVENUE
    // ========================================================

    const revenueMap = {{}};


    data.forEach(
        app => {{

            const category =
                app.Category || "Unknown";

            if (!revenueMap[category])
                revenueMap[category] = 0;

            revenueMap[category] +=
                Number(
                    app.Revenue || 0
                );

        }}
    );


    const revenueData =
        Object.entries(
            revenueMap
        )
        .sort(
            (a,b) => b[1] - a[1]
        )
        .slice(0,10);


    Plotly.react(
        "categoryRevenue",
        [
            {{
                x:
                    revenueData.map(
                        x => x[0]
                    ),

                y:
                    revenueData.map(
                        x => x[1]
                    ),

                type: "bar"

            }}
        ],
        {{
            ...plotLayout,

            title:
                "Estimated Revenue by Category",

            xaxis:
                {{
                    tickangle: -45
                }},

            yaxis:
                {{
                    title:
                        "Estimated Revenue"
                }}

        }},
        {{responsive:true}}
    );


    // ========================================================
    // RATING DISTRIBUTION
    // ========================================================

    const ratings =
        data
        .map(
            app =>
                Number(app.Rating)
        )
        .filter(
            x => isFinite(x)
        );


    Plotly.react(
        "ratingDistribution",
        [
            {{
                x: ratings,

                type: "histogram",

                nbinsx: 20

            }}
        ],
        {{
            ...plotLayout,

            title:
                "Rating Distribution",

            xaxis:
                {{
                    title:
                        "Rating"
                }},

            yaxis:
                {{
                    title:
                        "Number of Apps"
                }}

        }},
        {{responsive:true}}
    );


    // ========================================================
    // REVIEWS VS RATING
    // ========================================================

    Plotly.react(
        "reviewsRating",
        [
            {{
                x:
                    data.map(
                        app =>
                            Number(
                                app.Reviews || 0
                            )
                    ),

                y:
                    data.map(
                        app =>
                            Number(
                                app.Rating || 0
                            )
                    ),

                mode: "markers",

                type: "scatter",

                text:
                    data.map(
                        app =>
                            app.App
                    ),

                marker: {{

                    size: 7,

                    opacity: 0.55

                }}

            }}
        ],
        {{
            ...plotLayout,

            title:
                "Reviews vs Rating",

            xaxis: {{

                title:
                    "Reviews",

                type:
                    "log"

            }},

            yaxis: {{

                title:
                    "Rating",

                range:
                    [0,5]

            }}

        }},
        {{responsive:true}}
    );


    // ========================================================
    // SENTIMENT
    // ========================================================

    const sentimentMap = {{

        "Positive": 0,

        "Neutral": 0,

        "Negative": 0,

        "Unknown": 0

    }};


    data.forEach(
        app => {{

            const sentiment =
                app.Sentiment ||
                "Unknown";

            if (
                sentimentMap[
                    sentiment
                ] === undefined
            )
                sentimentMap[
                    sentiment
                ] = 0;

            sentimentMap[
                sentiment
            ]++;

        }}
    );


    Plotly.react(
        "sentimentChart",
        [
            {{

                labels:
                    Object.keys(
                        sentimentMap
                    ),

                values:
                    Object.values(
                        sentimentMap
                    ),

                type: "pie",

                hole: 0.45

            }}
        ],
        {{
            ...plotLayout,

            title:
                "User Sentiment Distribution"

        }},
        {{responsive:true}}
    );


    // ========================================================
    // RATING GROUP
    // ========================================================

    const ratingGroupMap = {{}};


    data.forEach(
        app => {{

            const group =
                app.Rating_Group ||
                "Unknown";

            if (!ratingGroupMap[group])
                ratingGroupMap[group] = 0;

            ratingGroupMap[group]++;

        }}
    );


    Plotly.react(
        "ratingGroupChart",
        [
            {{

                x:
                    Object.keys(
                        ratingGroupMap
                    ),

                y:
                    Object.values(
                        ratingGroupMap
                    ),

                type: "bar"

            }}
        ],
        {{
            ...plotLayout,

            title:
                "App Rating Groups"

        }},
        {{responsive:true}}
    );


    // ========================================================
    // TOP APPS
    // ========================================================

    const topApps =
        data
        .map(
            app => {{

                const successScore =
                    Number(
                        app.Rating || 0
                    ) *
                    Math.log1p(
                        Number(
                            app.Reviews || 0
                        )
                    );


                return {{

                    ...app,

                    SuccessScore:
                        successScore

                }};

            }}
        )
        .sort(
            (a,b) =>
                b.SuccessScore -
                a.SuccessScore
        )
        .slice(0,10);


    const table =
        document.getElementById(
            "topAppsTable"
        );


    table.innerHTML = "";


    topApps.forEach(
        (app,index) => {{

            const row =
                document.createElement(
                    "tr"
                );


            row.innerHTML = `

                <td>
                    ${{index + 1}}
                </td>

                <td>
                    ${{app.App || "Unknown"}}
                </td>

                <td>
                    ${{app.Category || "Unknown"}}
                </td>

                <td>
                    ${{Number(app.Rating || 0).toFixed(1)}}
                </td>

                <td>
                    ${{formatNumber(
                        Number(
                            app.Reviews || 0
                        )
                    )}}
                </td>

                <td>
                    ${{formatNumber(
                        Number(
                            app.Installs || 0
                        )
                    )}}
                </td>

                <td>
                    ${{app.Type || "Unknown"}}
                </td>

                <td>
                    ${{app.SuccessScore.toFixed(2)}}
                </td>

            `;


            table.appendChild(row);

        }}
    );


    // ========================================================
    // FEATURE IMPORTANCE
    // ========================================================

    const fi =
        featureImportance
        .slice()
        .sort(
            (a,b) =>
                b.Importance -
                a.Importance
        )
        .slice(0,15)
        .reverse();


    Plotly.react(
        "featureImportance",
        [
            {{

                x:
                    fi.map(
                        x =>
                            x.Importance
                    ),

                y:
                    fi.map(
                        x =>
                            x.Feature
                    ),

                type: "bar",

                orientation: "h"

            }}
        ],
        {{
            ...plotLayout,

            title:
                "Top Predictive Features",

            xaxis: {{

                title:
                    "Importance"

            }},

            yaxis: {{

                automargin: true

            }}

        }},
        {{responsive:true}}
    );

}}


// ==========================================================
// FILTER EVENT LISTENERS
// ==========================================================

document
    .getElementById(
        "categoryFilter"
    )
    .addEventListener(
        "change",
        updateDashboard
    );


document
    .getElementById(
        "typeFilter"
    )
    .addEventListener(
        "change",
        updateDashboard
    );


document
    .getElementById(
        "contentFilter"
    )
    .addEventListener(
        "change",
        updateDashboard
    );


document
    .getElementById(
        "yearFilter"
    )
    .addEventListener(
        "change",
        updateDashboard
    );


document
    .getElementById(
        "installFilter"
    )
    .addEventListener(
        "change",
        updateDashboard
    );


document
    .getElementById(
        "ratingMin"
    )
    .addEventListener(
        "input",
        updateDashboard
    );


document
    .getElementById(
        "ratingMax"
    )
    .addEventListener(
        "input",
        updateDashboard
    );


// ==========================================================
// INITIAL DASHBOARD LOAD
// ==========================================================

updateDashboard();


</script>


</body>

</html>

"""




os.makedirs(
    "/content/Results",
    exist_ok=True
)

dashboard_path = (
    "/content/Results/"
    "Google_Play_Store_Interactive_Dashboard.html"
)


with open(
    dashboard_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html)




print("=" * 80)

print(
    "DASHBOARD CREATED SUCCESSFULLY"
)

print("=" * 80)

print(
    "\nDashboard location:"
)

print(dashboard_path)


print(
    "\nFeatures included:"
)

print("✓ KPI Cards")

print("✓ Category Filter")

print("✓ Type Filter")

print("✓ Content Rating Filter")

print("✓ Year Filter")

print("✓ Minimum Install Filter")

print("✓ Rating Range Filter")

print("✓ Installs by Category")

print("✓ Estimated Revenue by Category")

print("✓ Rating Distribution")

print("✓ Reviews vs Rating")

print("✓ User Sentiment")

print("✓ Rating Groups")

print("✓ Top Performing Apps")

print("✓ Success Score")

print("✓ Random Forest Feature Importance")

print("✓ Fully Interactive HTML")




display(
    IFrame(
        src=dashboard_path,
        width="100%",
        height=1000
    )
)

In [ ]:


from google.colab import files
import os

# Path of generated dashboard
dashboard_path = "/content/Results/Google_Play_Store_Interactive_Dashboard.html"

# Check whether file exists
if os.path.exists(dashboard_path):

    print("✅ Dashboard file found!")
    print("📁 File:", dashboard_path)

    # Download HTML file
    files.download(dashboard_path)

    print("\n✅ Download started successfully!")
    print("Open the downloaded HTML file in Chrome/Edge/Firefox.")

else:

    print("❌ Dashboard file not found.")
    print("Please run the dashboard-generation cell first.")

In [ ]:
import shutil

# Specify the directory path and the desired name for the zip file (without extension)
directory_path = '/content/Results'
output_filename = '/content/Results'

# Create a zip archive of the folder
shutil.make_archive(output_filename, 'zip', directory_path)

print("ZIP file created successfully!")